**1. Set up and Load data**

In [0]:
import pandas as pd       # calling in the data manipulation library for usage

orders_df_spark=spark.read.table("mthimunye.default.orders")
customers_df_spark=spark.read.table("mthimunye.default.customers")
payments_df_spark=spark.read.table("mthimunye.default.payments")
products_df_spark=spark.read.table("mthimunye.default.products")

**1.1 Convert pyspark into pandas dataframe**

In [0]:
orders_df=orders_df_spark.toPandas()
customers_df=customers_df_spark.toPandas()
payments_df=payments_df_spark.toPandas()
products_df=products_df_spark.toPandas()

display(orders_df)
display(customers_df.head(5))
display(payments_df.head(5))
display(products_df)

**2. Data Interview**

**2.1 How many rows and columns are there**

In [0]:
orders_df.info()

note: orderdate dtype is incorrect (change from object to datetime64)

In [0]:
customers_df.info()

note: age dtype is incorrect (change from float to interger)

In [0]:
payments_df.info()

note: orderdate dtype is incorrect (change from object to datetime64)


In [0]:
products_df.info()

**2.2 Statistical summary of numeric columns**

In [0]:
orders_df.describe()

Anomalies in Quantity: The minimum quantity ordered is -2.000000, which indicates that there are negative values (likely returns, refunds, or data recording errors) that will need to be cleaned or handled in your data preparation step.

In [0]:
customers_df.describe()

In [0]:
payments_df.describe()

In [0]:
products_df.describe()

**3. Clean and Prepare**

In [0]:
orders_df.isnull().sum()

In [0]:
customers_df.isnull().sum()

In [0]:
payments_df.isnull().sum()

In [0]:
pd.crosstab(orders_df["PaymentMethod"].fillna("Missing"),orders_df["Status"].fillna("Missing"))

In [0]:
# Convert OrderDate dtype to datetime in orders_df
orders_df['OrderDate'] = pd.to_datetime(orders_df['OrderDate'])

# Convert PaymentDate dtype to datetime in payments_df
payments_df['PaymentDate'] = pd.to_datetime(payments_df['PaymentDate'])


In [0]:
#checking if dtype is successfully corrected
orders_df.info()

In [0]:
# Check for duplicate rows in each dataframe
orders_df.duplicated().sum()

In [0]:
customers_df.duplicated().sum()

In [0]:
payments_df.duplicated().sum()

In [0]:
products_df.duplicated().sum()

In [0]:
customers_df['City'].unique()

In [0]:
customers_df['City'] = customers_df['City'].replace({
    'tehran': 'Tehran',
    'Mashad': 'Mashhad'})

In [0]:
customers_df['City'].unique()

**3.1 Nulls Handling**

In [0]:
# Fill missing discount with 0 (no discount applied)

orders_df['Discount'] = orders_df['Discount'].fillna(0.0)

In [0]:
# Fill missing payment methods with 'Unknown'

orders_df['PaymentMethod'] = orders_df['PaymentMethod'].fillna('Unknown')

In [0]:
# Fill missing customer ages with the median age

customers_df['Age'] = customers_df['Age'].fillna(customers_df['Age'].median())

In [0]:
#Fill missing customer cities with 'Unknown'

customers_df['City'] = customers_df['City'].fillna('Unknown')

**3.2 Duplicates Handling**

In [0]:
orders_df.duplicated()


In [0]:
orders_df = orders_df.drop_duplicates()

In [0]:
orders_df.duplicated().sum()

**4. Joining Tables**

**4.1 Orders + Products**

In [0]:
orders_products_df = orders_df.merge(products_df, on="ProductID", how="inner")

display(orders_products_df)

**4.2 + Customers**

In [0]:
orders_customers_df = orders_df.merge(customers_df, on="CustomerID", how="inner")

display(orders_customers_df)

**4.3 + Payments**

In [0]:
orders_payments_df= orders_df.merge(payments_df, on="OrderID", how="inner")

display(orders_payments_df)

**5. Creating Tables**

*revenues should be calculated using 'Paid' order status only eliminating 'unpaid', 'returns'.*

In [0]:

completed_orders = orders_products_df[
    orders_products_df["Status"] == "Completed"
].copy()

completed_orders["Revenue"] = (
    completed_orders["Quantity"]
    * completed_orders["UnitPrice"]
    * (1 - completed_orders["Discount"])
)
display(completed_orders)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

revenues should be calculated using 'Paid' order status only eliminating 'unpaid', 'returns'. therefore the above table must be adjusted

In [0]:
completed_orders["Year"] = completed_orders["OrderDate"].dt.year

completed_orders["Month"] = completed_orders["OrderDate"].dt.month_name()

display(completed_orders)

In [0]:
orders_customers_products= completed_orders.merge(customers_df, on="CustomerID", how="inner")

display(orders_customers_products)


In [0]:
category_units = (
    orders_products_df[orders_products_df['Status'] == 'Completed']
    .groupby('Category', as_index=False)['Quantity']
    .sum()
    .sort_values('Quantity', ascending=False)
)

display(category_units)

**5. Visualisation - Using Plotly Express**

In [0]:
!pip install plotly
 

In [0]:
import plotly.express as px

In [0]:
monthly_revenue = (
    completed_orders[completed_orders['Status'] == 'Completed']
    .groupby(['Month'], as_index=False)['Revenue']
    .sum()
)

fig = px.line(
    monthly_revenue,
    x='Month',
    y='Revenue',
    markers=True,
    title='Monthly Revenue Trend'
)

fig.show()

In [0]:
top_products = (
    completed_orders[completed_orders['Status'] == 'Completed']
    .groupby('Category', as_index=False)['Revenue']
    .sum()
    .sort_values('Revenue', ascending=False)
)

fig = px.bar(
    top_products,
    x='Category',
    y='Revenue',
    title='Category by Revenue'
)

fig.show()

In [0]:
city_revenue = (
    orders_customers_products[orders_customers_products['Status'] == 'Completed']
    .groupby('City', as_index=False)['Revenue']
    .sum()
    .sort_values('Revenue', ascending=False)
)

fig = px.bar(
    city_revenue,
    x='City',
    y='Revenue',
    title='Revenue by City'
)

fig.show()

In [0]:
import plotly.express as px

segment_revenue = (
    orders_customers_products[orders_customers_products['Status'] == 'Completed']
    .groupby('CustomerSegment', as_index=False)['Revenue']
    .sum()
)

fig = px.pie(
    segment_revenue,
    names='CustomerSegment',
    values='Revenue',
    title='Revenue Contribution by Customer Segment',
    hole=0.3
)

fig.show()

In [0]:
payment_status = pd.crosstab(
    orders_payments_df['PaymentMethod'],
    orders_payments_df['Status'],
    normalize='index'
).reset_index()

payment_status = payment_status.melt(
    id_vars='PaymentMethod',
    var_name='Status',
    value_name='Percentage'
)

payment_status['Percentage'] = payment_status['Percentage'] * 100

display(payment_status)


In [0]:
fig = px.bar(
    payment_status,
    x='PaymentMethod',
    y='Percentage',
    color='Status',
    barmode='stack',
    title='Payment Method Performance by Transaction Status',
    labels={'Percentage': 'Percentage of Transactions (%)'}
)

fig.show()

In [0]:
discount_summary = (
    completed_orders[
       completed_orders['Status'] == 'Completed'
    ]
    .groupby('Discount', as_index=False)
    .agg(
        Total_Revenue=('Revenue', 'sum'),
        Orders=('OrderID', 'nunique')
    )
    .sort_values('Discount')
)

display(discount_summary)

In [0]:
fig = px.bar(
    discount_summary,
    x='Discount',
    y='Total_Revenue',
    title='Total Revenue by Discount Percentage',
    labels={
        'DiscountPercentage': 'Discount (%)',
        'Total_Revenue': 'Total Revenue'
    }
)

fig.show()